## 1. Important Imports

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pandas as pd 
import numpy as np 
import seaborn as se 
import matplotlib as plt
import random 
import plotly.express as px

## 2. Createting PySpark Session

In [0]:
spark = SparkSession.builder.appName('Data_Analysis_with_PySpark').getOrCreate()

## 3. Generating Data

In [0]:
names = [
    "Alice", "Bob", "Charlie", "David", "Eve", "Fiona", "George", "Hannah",
    "Ivy", "Jack", "Kaitlyn", "Liam", "Olivia", "Liam", "Emma", "Noah", 
    "Ava", "Oliver", "Charlotte", "Elijah", "Sophia", "James", "Amelia", 
    "Benjamin", "Isabella", "Lucas", "Mia", "Mason", "Harper", "Ethan", 
    "Evelyn", "Alexander", "Abigail", "Henry", "Ella", "Jackson", "Scarlett", 
    "Aiden", "Grace", "Samuel", "Lily", "Sebastian"
]
genders = ["Male", "Female", None]
subjects = ["Math", "Science", "History", "English", "Art", "PE", None]
cities = [
    "New York", "Los Angeles", "Chicago", "Houston", 
    "Bangalore", "Hajipur", "Sitamardhi", "MP", None
]
states = ["NY", "CA", "IL", "TX", "Bihar", "Karnataka", "Sitamardhi", None]
countries = ["USA", "India", "Pakistan", "Nepal", "China", None]
graduated_status = ["Yes", "No", None]

data = [
    (
        i, 
        random.choice(names),  # student_name
        random.choice([random.randint(18, 25), None]),  # age
        random.choice(genders),  # gender
        random.choice(subjects),  # subject
        random.choice([random.randint(50, 100), None]),  # marks
        random.choice(cities),  # city
        random.choice(states),  # state
        random.choice(countries),  # country
        random.choice(graduated_status),  # graduated
    )
    for i in range(1, 501)
]

## 4. Creating a DATAFRAME

In [0]:
schema = StructType([
    StructField("student_id", IntegerType(), True),
    StructField("student_name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("subject", StringType(), True),
    StructField("marks", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("country", StringType(), True),
    StructField("graduated", StringType(), True),
])

df = spark.createDataFrame(data,schema=schema)
df.show(5)

+----------+------------+---+------+-------+-----+----------+-----+--------+---------+
|student_id|student_name|age|gender|subject|marks|      city|state| country|graduated|
+----------+------------+---+------+-------+-----+----------+-----+--------+---------+
|         1|      Amelia| 19|  null|   null| null|Sitamardhi|   NY|   India|       No|
|         2|        Emma| 23|  Male|     PE|   80|  New York|   NY|     USA|       No|
|         3|    Benjamin| 25|  Male|     PE|   73|   Houston|   IL|     USA|     null|
|         4|       David| 24|Female|     PE| null|   Houston|   IL|Pakistan|       No|
|         5|      Oliver| 18|  null|English| null|   Houston|Bihar|   China|      Yes|
+----------+------------+---+------+-------+-----+----------+-----+--------+---------+
only showing top 5 rows



## 5.OverView of DataFrame

In [0]:
df.show(10)

+----------+------------+----+------+-------+-----+----------+-----+--------+---------+
|student_id|student_name| age|gender|subject|marks|      city|state| country|graduated|
+----------+------------+----+------+-------+-----+----------+-----+--------+---------+
|         1|      Amelia|  19|  null|   null| null|Sitamardhi|   NY|   India|       No|
|         2|        Emma|  23|  Male|     PE|   80|  New York|   NY|     USA|       No|
|         3|    Benjamin|  25|  Male|     PE|   73|   Houston|   IL|     USA|     null|
|         4|       David|  24|Female|     PE| null|   Houston|   IL|Pakistan|       No|
|         5|      Oliver|  18|  null|English| null|   Houston|Bihar|   China|      Yes|
|         6|   Alexander|null|  Male|Science|   95|  New York|   IL|Pakistan|      Yes|
|         7|     Kaitlyn|  24|  null|   Math|   86|  New York|Bihar|   Nepal|       No|
|         8|         Eve|  24|  null|     PE|   87|   Chicago|   TX|   China|     null|
|         9|    Scarlett|  21|Fe

In [0]:
df.describe().show()

+-------+-----------------+------------+------------------+------+-------+-----------------+----------+-----+-------+---------+
|summary|       student_id|student_name|               age|gender|subject|            marks|      city|state|country|graduated|
+-------+-----------------+------------+------------------+------+-------+-----------------+----------+-----+-------+---------+
|  count|              500|         500|               246|   343|    431|              250|       443|  443|    428|      331|
|   mean|            250.5|        null| 21.54471544715447|  null|   null|           76.136|      null| null|   null|     null|
| stddev|144.4818327679989|        null|2.2524436889210824|  null|   null|14.76120172293641|      null| null|   null|     null|
|    min|                1|     Abigail|                18|Female|    Art|               50| Bangalore|Bihar|  China|       No|
|    max|              500|      Sophia|                25|  Male|Science|              100|Sitamardhi| 

In [0]:
df.dtypes

Out[7]: [('student_id', 'int'),
 ('student_name', 'string'),
 ('age', 'int'),
 ('gender', 'string'),
 ('subject', 'string'),
 ('marks', 'int'),
 ('city', 'string'),
 ('state', 'string'),
 ('country', 'string'),
 ('graduated', 'string')]

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- graduated: string (nullable = true)



In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,157,69,0,57,57,72,169


## 6. Replacing Null Values

In [0]:
df = df.fillna({
    'gender' : 'UniSex',
    'subject' : 'Hindi',
    'city' : 'Kalitand',
    'State' : 'Others',
    'Country' : 'India',
    'graduated' : 'Failed'
})

In [0]:
## Checking Missing Values
str_column = ['student_name','gender','subject','city','state','country','graduated']
num_column = ['student_id','age','marks']

missing_val_dict = {}
for index,column in enumerate(df.columns):
    if column in str_column:
        missing_str_count = df.filter(col(column).eqNullSafe(None)\
            | col(column).isNull()
            ).count()
        missing_val_dict.update({column:missing_str_count})
    if column in num_column:
        missing_num_count = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val_dict.update({column:missing_num_count})
missing_df = pd.DataFrame.from_dict([missing_val_dict])
missing_df




,student_id,student_name,age,gender,subject,marks,city,state,country,graduated
0,0,0,0,0,0,0,0,0,0,0


## 7. Checking duplicate Values in each column

In [0]:
for index,column in enumerate(df.columns):
    print(f"checking duplicates in the column: {column}")
    duplicate_count = df.groupBy(column).count().filter("count > 1 ")
    duplicate_count.show()


checking duplicates in the column: student_id
+----------+-----+
|student_id|count|
+----------+-----+
+----------+-----+

checking duplicates in the column: student_name
+------------+-----+
|student_name|count|
+------------+-----+
|       Lucas|   10|
|       Grace|   10|
|         Ivy|    8|
|    Isabella|    9|
|       James|    9|
|    Benjamin|   10|
|         Ava|   10|
|        Ella|   16|
|      Evelyn|    9|
|        Noah|   13|
|       Mason|   11|
|     Charlie|   14|
|         Mia|    8|
|        Liam|   30|
|      Samuel|   10|
|   Alexander|   15|
|       Aiden|   11|
|       Alice|   10|
|     Kaitlyn|   12|
|         Eve|   10|
+------------+-----+
only showing top 20 rows

checking duplicates in the column: age
+----+-----+
| age|count|
+----+-----+
|  22|   36|
|null|  254|
|  20|   28|
|  19|   28|
|  23|   35|
|  25|   31|
|  24|   26|
|  21|   32|
|  18|   30|
+----+-----+

checking duplicates in the column: gender
+------+-----+
|gender|count|
+------+-----+
|Fe

In [0]:
duplicate_counts = []
# Loop through each column in the DataFrame
for column in df.columns:
    # Group by the column and count duplicates (count > 1)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,51
6,city,9
7,state,8
8,country,5
9,graduated,3


In [0]:
# Loop through each column in the DataFrame
for column in df.columns:
    print(column)
    dup_count = df.groupBy(column).count().filter("count > 1")
    # Get the number of duplicate values
    count = dup_count.count()
    # Append the result as a tuple (column name, duplicate count)
    duplicate_counts.append((column, count))
# Convert the list to a Pandas DataFrame
dup_df = pd.DataFrame(duplicate_counts, columns=['Column_Name', "Dup_count"])
# Show the Pandas DataFrame
dup_df


student_id
student_name
age
gender
subject
marks
city
state
country
graduated


,Column_Name,Dup_count
0,student_id,0
1,student_name,41
2,age,9
3,gender,3
4,subject,7
5,marks,51
6,city,9
7,state,8
8,country,5
9,graduated,3


## 8.Descriptive Statistics and Basic Summarization

In [0]:
df.printSchema()

root
 |-- student_id: integer (nullable = true)
 |-- student_name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- subject: string (nullable = false)
 |-- marks: integer (nullable = true)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- country: string (nullable = false)
 |-- graduated: string (nullable = false)



In [0]:
# What are the central tendencies (mean, median, mode) of marks and age?
df.groupBy("subject").agg(
    mean("marks").alias("marks_mean"),
    mean("age").alias("mean_age")
).show()



+-------+-----------------+------------------+
|subject|       marks_mean|          mean_age|
+-------+-----------------+------------------+
|Science|73.94444444444444| 21.70967741935484|
|    Art|79.57894736842105| 21.57894736842105|
|   Math|74.74285714285715|21.404761904761905|
|English|72.26315789473684|  21.2972972972973|
|History|             80.2|21.884615384615383|
|  Hindi|73.78947368421052|21.444444444444443|
|     PE|79.31428571428572| 21.63888888888889|
+-------+-----------------+------------------+



In [0]:
# What is the spread of marks in each subject?
df.groupBy("subject").agg(
    min("marks").alias("min_marks"),
    max("marks").alias("max_marks"),
).show()

+-------+---------+---------+
|subject|min_marks|max_marks|
+-------+---------+---------+
|Science|       51|       99|
|    Art|       51|      100|
|   Math|       53|      100|
|English|       50|       99|
|History|       54|       99|
|  Hindi|       50|       98|
|     PE|       57|       98|
+-------+---------+---------+



In [0]:
# What is the standard deviation of marks for different cities or countries?
df.groupBy("city").agg(
    stddev("marks").alias("stddev_marks")
).show()

+-----------+------------------+
|       city|      stddev_marks|
+-----------+------------------+
|  Bangalore|13.779419936742746|
|Los Angeles|13.623864598919303|
| Sitamardhi|16.574780038761702|
|    Chicago|14.806295332110443|
|    Hajipur|13.061726365690484|
|    Houston|15.723911069143863|
|   New York|12.885535962350847|
|         MP|15.346186326987194|
|   Kalitand|16.208412743947534|
+-----------+------------------+



In [0]:
from pyspark.sql import functions as F

# Calculate the distribution of students in each subject
df.groupBy("subject").agg(
    F.count("student_id").alias("stu_count")  # Count the number of students in each subject
).show()


+-------+---------+
|subject|stu_count|
+-------+---------+
|Science|       73|
|    Art|       78|
|   Math|       73|
|English|       83|
|History|       55|
|  Hindi|       69|
|     PE|       69|
+-------+---------+



In [0]:
# How many students belong to each gender and how does their academic performance differ?
df.groupBy("gender").agg(
    F.count("*").alias("stu_count"),       # Count of students per gender
    F.avg("marks").alias("avg_marks"),     # Average marks per gender
    F.stddev("marks").alias("std_marks")   # Standard deviation of marks per gender
).show()


+------+---------+-----------------+------------------+
|gender|stu_count|        avg_marks|         std_marks|
+------+---------+-----------------+------------------+
|Female|      180|76.38297872340425|14.924978257204835|
|UniSex|      157|76.67105263157895| 14.35677601496449|
|  Male|      163|          75.3375|15.095429562317399|
+------+---------+-----------------+------------------+



In [0]:
# What is the distribution of students across different cities, states, and countries?
df.groupBy("city","state","country").count().show()

+----------+----------+--------+-----+
|      city|     state| country|count|
+----------+----------+--------+-----+
|Sitamardhi|Sitamardhi|   India|    3|
|  Kalitand|        IL|Pakistan|    1|
|  New York| Karnataka|   India|    1|
|  Kalitand|Sitamardhi|Pakistan|    2|
|Sitamardhi|        NY|   India|    2|
|  New York|     Bihar|   Nepal|    4|
|   Houston|Sitamardhi|   Nepal|    3|
|   Chicago|        NY|   China|    3|
| Bangalore|        CA|     USA|    1|
|        MP|Sitamardhi|   India|    3|
| Bangalore|     Bihar|     USA|    2|
|  Kalitand|        NY|   India|    6|
|  New York|        TX|   India|    3|
|   Houston|    Others|Pakistan|    1|
| Bangalore|    Others|   India|    1|
|Sitamardhi|        TX|Pakistan|    1|
| Bangalore| Karnataka|Pakistan|    2|
|   Houston| Karnataka|   India|    5|
|   Hajipur|    Others|   China|    2|
|   Hajipur|    Others|   Nepal|    1|
+----------+----------+--------+-----+
only showing top 20 rows



In [0]:
# How many students are marked as graduated versus those who are not? What is the graduation rate?
graduated_status = df.groupBy("graduated").agg(F.count("*").alias("stu_count"))
total_stu = df.count()

graduated_status.withColumn("graducation_perc",col("stu_count")/total_stu * 100).show()


+---------+---------+------------------+
|graduated|stu_count|  graducation_perc|
+---------+---------+------------------+
|       No|      165|              33.0|
|   Failed|      169|33.800000000000004|
|      Yes|      166|              33.2|
+---------+---------+------------------+



In [0]:
graduated_status.show()

+---------+---------+
|graduated|stu_count|
+---------+---------+
|       No|      165|
|   Failed|      169|
|      Yes|      166|
+---------+---------+



##  9.Exploratory Data Analysis (EDA) : Marks Distribution

In [0]:
# What is the distribution of marks within each subject? Are there specific subjects with higher or lower marks?

marks_dis = df.groupBy("subject").agg(F.sum("marks").alias("total_marks"))
marks_dis = marks_dis.toPandas()

fig_bar = px.bar(
    marks_dis, x= 'subject',y = 'total_marks', # define x and y asis 
    title = "Total Marks Distribution By Subject", # Add Title into the Graph
    labels={"total_marks": "Total Marks", "subject": "Subject"},# display labels
    text='total_marks' , # Display total marks on the bars
        )
fig_bar.show()

fig_pie = px.pie(
    marks_dis,
    names = 'subject',
    values = 'total_marks',
    title = "Total Marks Distribution By Subject",
    labels = {"subject": "Subject", "total_marks": "Total Marks"},
    color_discrete_sequence = px.colors.qualitative.Pastel 
)
fig_pie.show()

In [0]:
# How do marks vary across different age groups or genders?
marks_by_age = df.filter(col("age").isNotNull()).groupBy("age").agg(sum("marks").alias("total_marks"))
marks_by_age = marks_by_age.toPandas()
#print(marks_by_age)

fig_bar = px.bar(
    marks_by_age,
    x = 'age',
    y = 'total_marks',
    title = 'Marks Distribution By Age',
    labels = {'total_marks' : 'Total Marks' , 'subject':'Subjet'},
    text = 'total_marks'
)
fig_bar.show()

fig_pie = px.pie(
    marks_by_age,
    names = 'age',
    values = 'total_marks',
    title = "Total Marks Distribution By Subject",
    labels = {"age": "Age", "total_marks": "Total Marks"},
    color_discrete_sequence = px.colors.qualitative.Pastel 
)
fig_pie.show()

marks_by_gender = df.filter(col("gender").isNotNull()).groupBy("gender").agg(sum("marks").alias("total_marks"))
marks_by_gender = marks_by_gender.toPandas()

fig_bar_by_gender = px.bar(
    marks_by_gender,
    x =  'gender',
    y = 'total_marks',
    title = 'Marks Distribution By Gender',
    labels = {"gender" : "Gender","total_marks" : "Total_Marks"},
    text = 'total_marks'
)
fig_bar_by_gender.show()

fig_pie_by_gender = px.pie(
    marks_by_gender,
    names = 'gender',
    values = 'total_marks',
    title = 'Marks Distribution By Gender',
    color_discrete_sequence = px.colors.qualitative.Antique
)

fig_pie_by_gender.show()



## 10 .Exploratory Data Analysis (EDA) : Creating Stacked Bar Graph

In [0]:
# How does the age distribution look? Is it skewed or normal? Are there specific age groups that perform better academically?

df = df.withColumn(
    "age_group",
    F.when(F.col("age") < 18, "Minor")
     .when((F.col("age") >= 18) & (F.col("age") <= 30), "Young Adult")
     .otherwise("Adult")
)
student_by_age_and_gender = df.groupBy("age_group", "gender").agg(F.count("*").alias("stu_count"))
student_by_age_and_gender = student_by_age_and_gender.toPandas()

fig_stacked_bar = px.bar(
    student_by_age_and_gender,
    x = 'age_group',
    y = 'stu_count',
    color = 'gender',
    title = 'Distribution By Age and Gender',
    text = 'stu_count',
    color_discrete_sequence = px.colors.qualitative.Pastel
)
fig_stacked_bar.show()



## WORKING WITH GRAPHS


## 11. Marks Distribution (Histogram)

In [0]:
df1 = df.toPandas() 
fig = px.histogram(
    df1,
    x = 'marks',
    nbins= 20,
    title= 'Marks Distribution'
)
fig.update_layout(xaxis_title = 'marks',yaxis_title = 'Frequency')
fig.show()


## 12.Average Marks by Subject (Bar Chart)

In [0]:
avg_marks_by_subject = df.groupBy("subject").agg(F.mean("marks").alias("Avg_marks"))
#avg_marks_by_subject.show()
avg_marks_by_subject = avg_marks_by_subject.toPandas()

fig1 = px.bar(
    avg_marks_by_subject,
    x = 'subject',
    y = 'Avg_marks',
    title= 'Avg Marks Distribution By Subject',
    text = 'Avg_marks',
    color_discrete_sequence= px.colors.qualitative.Pastel
)
fig1.show()


In [0]:
df1.columns

Out[30]: Index(['student_id', 'student_name', 'age', 'gender', 'subject', 'marks',
       'city', 'state', 'country', 'graduated', 'age_group'],
      dtype='object')


## 13. Marks vs Age (Scatter Plot)

In [0]:
fig = px.scatter(
    df1,
    x='age',
    y='marks',
    color='graduated',
    title= 'Marks Vs Age',
    labels= {'graduated':'Graduation_Status'},
)
fig.show()


## 14. Gender Distribution (Pie Chart)


## 15.Average Marks by City (Horizontal Bar Chart)


## 16.Graduation Status by Subject (Stacked Bar Chart)


## 17.Correlation Heatmap